# JORINOVA NEXUS — LIS auto-mapping (Track 2)

Fine-tunes `google/flan-t5-base` to read raw OCR'd lab-request text → extract `pid`, `family_name`, `gender`, `priority`, and the list of tests as JSON. Today's regex extractor scores 100% on the golden but only for forms that look like the templates. The model adds robustness to weird layouts the regex rejects.

## Before pressing Run All

1. **Runtime → Change runtime type → T4 GPU**
2. **Upload** two files to the Colab files panel:
   - `datasets/lis_mapping.jsonl`
   - `backend/training/golden/lis_mapping_golden.jsonl`

## What it does

1. Loads the data, splits 70/15/15 train/val/test (plus the locked golden eval)
2. Fine-tunes Flan-T5-base on text-to-JSON
3. Trains 3 epochs at 3e-4 (~30 min on T4)
4. Scores exact-match field accuracy on the locked golden
5. Exports a HF model directory for wrapping in a FastAPI route

## 1. Install + import

In [ ]:
!pip install -q transformers==4.45.0 datasets==3.0.0 accelerate==1.0.0 sentencepiece==0.2.0

In [ ]:
import json, random, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

random.seed(42); torch.manual_seed(42)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Load the data

In [ ]:
TRAIN_PATH  = 'lis_mapping.jsonl'        # 80 real LabRequests reverse-engineered to text
GOLDEN_PATH = 'lis_mapping_golden.jsonl'  # 10 hand-curated locked eval

def jload(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

train_all = jload(TRAIN_PATH)
print(f'training pool : {len(train_all)} examples')

if Path(GOLDEN_PATH).is_file():
    golden = jload(GOLDEN_PATH)
    GOLDEN_IS_LOCKED = True
    print(f'golden eval   : {len(golden)} examples  (locked, hand-curated)')
else:
    print()
    print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!')
    print('!! lis_mapping_golden.jsonl NOT FOUND.                            !!')
    print('!!                                                                !!')
    print('!! Upload backend/training/golden/lis_mapping_golden.jsonl        !!')
    print('!! to Colab alongside lis_mapping.jsonl for proper evaluation.    !!')
    print('!!                                                                !!')
    print('!! Falling back to a 15%% held-out slice of lis_mapping.jsonl.     !!')
    print('!! REPORTED FIELD-LEVEL ACCURACY WILL BE OPTIMISTIC.              !!')
    print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!')
    print()
    random.shuffle(train_all)
    n_hold = max(1, int(0.15 * len(train_all)))
    golden    = train_all[:n_hold]
    train_all = train_all[n_hold:]
    GOLDEN_IS_LOCKED = False
    print(f'held-out eval : {len(golden)} examples  (FALLBACK, optimistic)')
    print(f'training pool : {len(train_all)} examples')

print('sample row:')
print(json.dumps(train_all[0], indent=2)[:400])

## 3. Format as text-to-JSON pairs

Flan-T5 is a seq2seq model — input is the raw OCR text, target is the compact JSON string of the extracted fields.

In [ ]:
TASK_PREFIX = 'Extract patient and tests as JSON: '

def to_pair(r):
    # Compose the target JSON in a deterministic order so the model has a stable label
    target = {
        'pid':        r.get('pid', ''),
        'family_name': r.get('family_name', ''),
        'gender':     r.get('gender', ''),
        'priority':   r.get('priority', 'routine'),
        'tests':      r.get('tests', []),
    }
    return {
        'input':  TASK_PREFIX + r.get('raw_text', ''),
        'target': json.dumps(target, ensure_ascii=False, separators=(',', ':')),
    }

random.shuffle(train_all)
n = len(train_all)
train_pool = train_all[: int(0.7*n)]
val_pool   = train_all[int(0.7*n) : int(0.85*n)]
test_pool  = train_all[int(0.85*n) :]

train_ds = Dataset.from_list([to_pair(r) for r in train_pool])
val_ds   = Dataset.from_list([to_pair(r) for r in val_pool])
test_ds  = Dataset.from_list([to_pair(r) for r in test_pool])
print(f'train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}  golden: {len(golden)}')

## 4. Load Flan-T5 + tokenise

In [ ]:
MODEL_ID = 'google/flan-t5-base'  # 250M params, fits easily on T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

def tokenize(batch):
    inp = tokenizer(batch['input'], max_length=512, truncation=True, padding=False)
    tgt = tokenizer(batch['target'], max_length=256, truncation=True, padding=False)
    inp['labels'] = tgt['input_ids']
    return inp

train_tok = train_ds.map(tokenize, batched=True, remove_columns=['input','target'])
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=['input','target'])

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
print('tokenised. train tokens per sample:', sum(len(x['input_ids']) for x in train_tok) // len(train_tok))

## 5. Train

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir='./out',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    bf16=True,
    predict_with_generate=True,
    generation_max_length=256,
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=collator,
)
trainer.train()

## 6. Score on the locked golden set

Field-level exact match for each of `pid`, `family_name`, `gender`, `priority`, and set-equality for `tests`. Target ≥ 95% per field on novel layouts. The regex stays the fast path; this model is the 'weird forms' fallback.

In [ ]:
model.eval()

def predict_json(raw_text):
    enc = tokenizer(TASK_PREFIX + raw_text, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    out = model.generate(**enc, max_new_tokens=256, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

field_scores = {k: [0, 0] for k in ('pid','family_name','gender','priority','tests')}
for r in golden:
    pred_str = predict_json(r['raw_text'])
    try:
        pred = json.loads(pred_str)
    except json.JSONDecodeError:
        pred = {}
    for k in field_scores:
        field_scores[k][1] += 1
        ref = r.get(k) if k != 'tests' else set(r.get('tests', []))
        got = pred.get(k) if k != 'tests' else set(pred.get('tests', []))
        if ref == got:
            field_scores[k][0] += 1

label = 'on LOCKED GOLDEN' if globals().get('GOLDEN_IS_LOCKED', True) else 'on FALLBACK SPLIT (optimistic)'
print(f'Per-field accuracy {label}:')
for k, (ok, tot) in field_scores.items():
    print(f'  {k:12s}: {ok}/{tot} = {ok/tot:.0%}')
if not globals().get('GOLDEN_IS_LOCKED', True):
    print('
WARNING: scores above are OPTIMISTIC. Upload lis_mapping_golden.jsonl for a trustworthy number.')


## 7. Save + wire-up instructions

Flan-T5 doesn't need GGUF/Ollama — just save the HF model dir and load it in a small FastAPI route on the backend.

In [ ]:
model.save_pretrained('./jorinova-lis-mapping')
tokenizer.save_pretrained('./jorinova-lis-mapping')
print('Saved ./jorinova-lis-mapping/')
print()
print('Next on your laptop:')
print('  1. Download jorinova-lis-mapping/ from the Colab files panel')
print('  2. Drop it in backend/ai_services/models/jorinova-lis-mapping/')
print('  3. Add a FastAPI route /api/v1/lis-mapping/extract_with_model that loads it')
print('  4. lis_mapping.py keeps regex as the fast path; calls the model only if regex returns empty')